# 7클래스 + 실제 녹음 학습 — GPU 서버 실행 (S15P31A307-241, 설계 CHART-003-real-drum-data-check 작업 12)

크래시·라이드를 모델이 직접 나눠 내는 7클래스 학습에 실제 녹음(ENST·MDB)을 더해 학습한다. 심벌을 이 모델에서 가져왔을 때 11/02 당시 서비스 조합(combo)보다 나은지 드럼 분리 입력으로 비교하려는 것이다.

| 학습 | 무엇 |
| --- | --- |
| `tom_7cls_real_full` | ENST·MDB 전체를 넣은 학습. 되돌리기용 combo 모델이 씀 |
| `tom_7cls_real_no_d1~3` | 채점용. ENST 드러머 하나와 MDB 묶음 하나를 빼고 학습해, 뺀 곡으로만 채점 |

**준비물.** 로컬 `AI/`에서 `uv run python scripts/pack_server_upload.py --code-only`로 만든 `data/upload/server/S15P31A307_code.zip`을 JupyterHub `ssaplash/upload/`에, 이 파일을 `ssaplash/notebooks/`에 올린다.

**순서.** 1 설정 → 1-1 상태 → 2 코드 풀기 → 3 학습 4개 동시 시작 → 4 학습 진행 확인(약 1~1.5시간) → 5 채점용 예측 시작(학습이 끝난 것부터, 여러 번 쳐도 됨) → 6 예측 진행 확인 → 7 결과 묶기. 3·5번은 백그라운드라 브라우저를 닫아도 돈다. 커널이 다시 시작됐으면 1번을 치고 이어 간다.

In [ ]:
# 1. 설정. 지정받은 GPU는 6번
import getpass
import glob
import os
import subprocess
import sys

GPU_ID = "6"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["MPLBACKEND"] = "Agg"
os.environ["OMP_NUM_THREADS"] = "1"       # 데이터 읽기 작업이 코어를 서로 뺏지 않게
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
HOME = os.path.expanduser("~")
WORK = f"{HOME}/ssaplash"
AI = f"{WORK}/S15P31A307/AI"
PY = sys.executable
VENV_PY = f"{AI}/.venv/bin/python"
ME = getpass.getuser()
UPLOAD = f"{WORK}/upload"
RESULTS = f"{WORK}/results"
RUN_DIR = f"{WORK}/runs/241-cym7-real"    # 학습용 코드 사본·대기열 상태
QUEUE = f"{VENV_PY} {AI}/scripts/train_queue.py"
PARALLEL = 4
CPUS = "0-95"
NUM_WORKERS = 8
TRAIN = {n: f"configs/train/real_check/tom_7cls_real_{n}.yaml" for n in ("full", "no_d1", "no_d2", "no_d3")}
OUT = {n: f"data/models/adtof_ft/tom_7cls_real_{n}" for n in TRAIN}
RES = "data/results/sep_compare"
DS = "configs/bench/datasets"
SEPARATOR = "configs/bench/models/msst/bs_roformer_sw.yaml"
# 채점용 예측 run 이름 → (학습 이름, 어댑터, 데이터셋, 기대 곡 수). 드럼은 run마다 BS-Roformer SW로 새로 뽑음
SCORE = {}
for k, (nm, ne) in zip((1, 2, 3), ((8, 21), (7, 22), (8, 21)), strict=True):
    adapter = f"configs/bench/transcribers/adtof_drumsep/sep_compare/cym7real_heldout_d{k}.yaml"
    SCORE[f"cym7real_mdb_d{k}"] = (f"no_d{k}", adapter, f"{DS}/real_check/mdb_fold{k}.yaml", nm)
    SCORE[f"cym7real_enst_d{k}"] = (f"no_d{k}", adapter, f"{DS}/real_check/enst_minus_one_d{k}.yaml", ne)
SLOG = {run: f"{AI}/sep_compare_{run}.log" for run in SCORE}
os.makedirs(RESULTS, exist_ok=True)
%cd $AI
print("학습", list(TRAIN), "· 채점 run", len(SCORE), "개")
!nvidia-smi -i $GPU_ID --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv
!ls -la $UPLOAD/S15P31A307_code.zip
# 확인: '학습 ['full', 'no_d1', 'no_d2', 'no_d3'] · 채점 run 6 개', GPU 한 줄(H200), 코드 zip 날짜가 오늘 올린 것

In [ ]:
# 1-1. 지금 상태 (여러 번 쳐도 됨)
!$QUEUE status --run-dir $RUN_DIR
!ps -u $ME -o pid,etime,pcpu,args | grep -E "[a]dtof-finetune|[t]rain_queue|[r]un-transcription" | cut -c1-150
!nvidia-smi -i $GPU_ID --query-gpu=memory.used,utilization.gpu --format=csv,noheader
# 확인: 3번 전이면 '대기열을 시작한 적 없음'. 3번 뒤에는 학습마다 '도는 중' / '끝남' / '실패' / '멈춤'

In [ ]:
# 2. 코드 덮어 풀기
!$PY -m zipfile -e $UPLOAD/S15P31A307_code.zip $WORK
%cd $AI
!ls configs/train/real_check/ | grep tom_7cls_real
!ls configs/bench/transcribers/adtof_drumsep/sep_compare/ | grep cym7real
!ls data/models/adtof_ft/ | head -20
# 확인: tom_7cls_real_full·no_d1·no_d2·no_d3 yaml 4개, cym7real_heldout_d1~3 yaml 3개.
#   없으면 zip이 예전 것. 로컬에서 zip을 다시 만들어 올림

In [ ]:
# 3. 학습 4개 동시 시작 (백그라운드, 한 번만. 다시 쳐도 대기열이 돌고 있으면 안 띄움. 멈춘 학습은 last.pth에서 이어 감)
runs = " ".join(f"--run tom_7cls_real_{n}={cfg}" for n, cfg in TRAIN.items())
!$QUEUE start --run-dir $RUN_DIR --zip $UPLOAD/S15P31A307_code.zip --shared-ai $AI --python $VENV_PY --num-workers $NUM_WORKERS --parallel $PARALLEL --cpus $CPUS $runs
# 확인: '코드 사본 풂: …/runs/241-cym7-real/…', '대기열 시작 pid …: tom_7cls_real_full → … , 동시 4개, …'.
#   '학습 설정을 못 찾음'이면 zip이 예전 것(2번 확인). 'taskset: command not found'면 1번 CPUS를 "" 로 바꾸고 다시

In [ ]:
# 4. 학습 진행 확인 (여러 번). 학습 하나에 약 1~1.5시간, 넷이 같이 돎
!$QUEUE status --run-dir $RUN_DIR
# 확인: 학습마다 '[train] 에폭 N 손실 … 검증 f1 …' 줄이 늘고, 끝나면 '끝남'.
#   '실패'면 RUN_DIR/status/<이름>.log 끝을 붙여 줌. '멈춤'이면 3번을 다시 침.
#   no_d1~3 중 '끝남'이 생기면 5번을 쳐도 됨(끝난 것만 채점 예측을 띄움)

In [ ]:
# 5. 채점용 예측 시작 (GPU, 백그라운드). 학습이 끝난 no_d* 것만 띄우고, 이미 띄운 run은 건너뜀. 여러 번 쳐도 됨
def alive(run):
    try:
        pid = int(open(SLOG[run] + ".pid").read())
        with open(f"/proc/{pid}/stat") as f:
            return f.read().rsplit(")", 1)[1].split()[0] != "Z"
    except (OSError, ValueError, IndexError):
        return False


def n_midi(run):
    return len(glob.glob(f"{RES}/{run}/midi/**/*.mid", recursive=True))


for run, (train, adapter, ds, n) in SCORE.items():
    if n_midi(run) >= n:
        print(run, "이미 끝남")
        continue
    if alive(run):
        print(run, "도는 중, 건너뜀")
        continue
    if not (os.path.isfile(f"{RUN_DIR}/status/tom_7cls_real_{train}.DONE") and os.path.isfile(f"{OUT[train]}/best.pth")):
        print(run, "학습이 아직 안 끝남, 건너뜀 (4번으로 확인 뒤 다시 침)")
        continue
    cmd = (f".venv/bin/ssaplash-bench run-transcription --models {adapter} --dataset {ds} --input separated "
           f"--separator {SEPARATOR} --out {RES} --run-id {run} --save-midi 9999 --device cuda --mlflow-uri none")
    proc = subprocess.Popen(["nohup", "nice", "-n", "19", "bash", "-c", cmd], cwd=AI, stdout=open(SLOG[run], "a"),
                            stderr=subprocess.STDOUT, env=dict(os.environ), start_new_session=True)
    with open(SLOG[run] + ".pid", "w") as f:
        f.write(str(proc.pid))
    print("시작.", run, "pid", proc.pid)
# 확인: 학습이 끝난 묶음마다 '시작. cym7real_mdb_dK …', '시작. cym7real_enst_dK …'. 나머지는 '학습이 아직 안 끝남'

In [ ]:
# 6. 채점용 예측 진행 확인 (여러 번)
for run, (_, _, _, n) in SCORE.items():
    got = n_midi(run)
    state = "끝" if got >= n else ("도는 중" if alive(run) else "대기 또는 멈춤")
    print(f"{run:18s} 예측 MIDI {got:3d} / {n}  {state}")
!grep -l -E "Traceback|out of memory" {AI}/sep_compare_cym7real_*.log 2>/dev/null || true
# 확인: 6줄이 모두 '끝'이고 4번에서 tom_7cls_real_full도 '끝남'이면 7번.
#   '대기 또는 멈춤'이면 5번을 다시 침. 마지막 grep에 로그 이름이 나오면 `!tail -30 <로그>` 결과를 붙여 줌

In [ ]:
# 7. 결과 묶기. results 폴더의 cym7_real_results.zip 을 내려받아 로컬 AI/data/results/ 에 둠
import zipfile

items = [f"{OUT[n]}/{f}" for n in TRAIN for f in ("best.pth", "log.csv", "config.yaml")]
missing = [p for p in items if not os.path.isfile(p)]
with zipfile.ZipFile(f"{RESULTS}/cym7_real_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in items:
        if os.path.isfile(p):
            z.write(p)
    for run in SCORE:
        for p in glob.glob(f"{RES}/{run}/**/*", recursive=True):
            if os.path.isfile(p) and not p.endswith(".wav"):
                z.write(p)
        if os.path.exists(SLOG[run]):
            z.write(SLOG[run], os.path.basename(SLOG[run]))
    for p in glob.glob(f"{RUN_DIR}/status/*"):
        z.write(p, f"train-queue-status-cym7real/{os.path.basename(p)}")
    names = z.namelist()
!ls -la $RESULTS/cym7_real_results.zip
print("가중치·로그 빠진 것:", missing or "없음", "· 예측 MIDI", sum(n.endswith(".mid") for n in names), "개")
# 확인: '가중치·로그 빠진 것: 없음 · 예측 MIDI 87 개'와 수 MB짜리 zip 한 줄